VectorBT is a tool that allow easily to backtest strategies with a couple of lines of Python code

In [3]:
import vectorbt as vbt

price = vbt.YFData.download('BTC-USD').get('Close')

pf = vbt.Portfolio.from_holding(price, init_cash=100)
pf.total_profit()

19099.868890273745

In [9]:
# Buy whenever 10-day SMA crosses above 50-day SMA and sell when opposite: 
fast_ma = vbt.MA.run(price, 10)
slow_ma = vbt.MA.run(price, 50) 
entries = fast_ma.ma_crossed_above(slow_ma)
exits = fast_ma.ma_crossed_below(slow_ma)

pf = vbt.Portfolio.from_signals(price, entries, exits, init_cash=100)
pf.total_profit()


34417.80960086067

In [2]:
# Let's generate 1000 strategies with random signals and test them on BTC and ETH: 

import numpy as np

symbols = ["BTC-USD", "ETH-USD"]
price = vbt.YFData.download(symbols, missing_index='drop').get('Close')

n = np.random.randint(10, 101, size=1000).tolist()
pf = vbt.Portfolio.from_random_signals(price, n=n, init_cash=100, seed=42)

mean_expectancy = pf.trades.expectancy().groupby(['randnx_n', 'symbol']).mean()
fig = mean_expectancy.unstack().vbt.scatterplot(xaxis_title='randnx_n', yaxis_title='mean_expectancy')
fig.show()

/Users/mark/Library/Python/3.10/lib/python/site-packages/vectorbt/data/base.py:527: UserWarning:

Symbols have mismatching index. Dropping missing data points.



In [3]:
#For fans of hyperparameter optimization: here is a snippet for testing 10,000 window combinations of a dual SMA crossover strategy on BTC, USD, and LTC:

symbols = ["BTC-USD", "ETH-USD", "LTC-USD"]
price = vbt.YFData.download(symbols, missing_index='drop').get('Close')

windows = np.arange(2, 101)
fast_ma, slow_ma = vbt.MA.run_combs(price, window=windows, r=2, short_names=['fast', 'slow'])
entries = fast_ma.ma_crossed_above(slow_ma)
exits = fast_ma.ma_crossed_below(slow_ma)

pf_kwargs = dict(size=np.inf, fees=0.001, freq='1D')
pf = vbt.Portfolio.from_signals(price, entries, exits, **pf_kwargs)

fig = pf.total_return().vbt.heatmap(
    x_level='fast_window', y_level='slow_window', slider_level='symbol', symmetric=True,
    trace_kwargs=dict(colorbar=dict(title='Total return', tickformat='%')))
fig.show()

/Users/mark/Library/Python/3.10/lib/python/site-packages/vectorbt/data/base.py:527: UserWarning:

Symbols have mismatching index. Dropping missing data points.

/Users/mark/Library/Python/3.10/lib/python/site-packages/jupyter_client/session.py:721: UserWarning:

Message serialization failed with:
Out of range float values are not JSON compliant
Supporting this message is deprecated in jupyter-client 7, please make sure your message is JSON-compliant

/Users/mark/Library/Python/3.10/lib/python/site-packages/jupyter_client/session.py:721: UserWarning:

Message serialization failed with:
Out of range float values are not JSON compliant
Supporting this message is deprecated in jupyter-client 7, please make sure your message is JSON-compliant

/Users/mark/Library/Python/3.10/lib/python/site-packages/jupyter_client/session.py:721: UserWarning:

Message serialization failed with:
Out of range float values are not JSON compliant
Supporting this message is deprecated in jupyter-client 7, please

In [4]:
#Digging into each strategy configuration is as simple as indexing with pandas: 
pf[(10, 20, 'ETH-USD')].stats()


Start                          2017-11-09 00:00:00+00:00
End                            2025-12-28 00:00:00+00:00
Period                                2972 days 00:00:00
Start Value                                        100.0
End Value                                    1618.326322
Total Return [%]                             1518.326322
Benchmark Return [%]                          816.941505
Max Gross Exposure [%]                             100.0
Total Fees Paid                               202.609579
Max Drawdown [%]                               70.734951
Max Drawdown Duration                 1095 days 00:00:00
Total Trades                                          80
Total Closed Trades                                   80
Total Open Trades                                      0
Open Trade PnL                                       0.0
Win Rate [%]                                       41.25
Best Trade [%]                                120.511071
Worst Trade [%]                

In [5]:
# The same for plotting:

pf[(10, 20, 'ETH-USD')].plot().show()

In [12]:
symbols = ["BTC-USD", "ETH-USD", "ADA-USD"]
price = vbt.YFData.download(symbols, period='6mo', missing_index='drop').get('Close')
bbands = vbt.BBANDS.run(price)

def plot(index, bbands):
    bbands = bbands.loc[index]
    fig = vbt.make_subplots(
        rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.15,
        subplot_titles=('%B', 'Bandwidth'))
    fig.update_layout(template='vbt_dark', showlegend=False, width=750, height=400)
    bbands.percent_b.vbt.ts_heatmap(
        trace_kwargs=dict(zmin=0, zmid=0.5, zmax=1, colorscale='Spectral', colorbar=dict(
            y=(fig.layout.yaxis.domain[0] + fig.layout.yaxis.domain[1]) / 2, len=0.5
        )), add_trace_kwargs=dict(row=1, col=1), fig=fig)
    bbands.bandwidth.vbt.ts_heatmap(
        trace_kwargs=dict(colorbar=dict(
            y=(fig.layout.yaxis2.domain[0] + fig.layout.yaxis2.domain[1]) / 2, len=0.5
        )), add_trace_kwargs=dict(row=2, col=1), fig=fig)
    return fig

vbt.save_animation('bbands.gif', bbands.wrapper.index, plot, bbands, delta=90, step=3, fps=3)

  0%|          | 0/32 [00:00<?, ?it/s]

/Users/mark/Library/Python/3.10/lib/python/site-packages/jupyter_client/session.py:721: UserWarning:

Message serialization failed with:
Out of range float values are not JSON compliant
Supporting this message is deprecated in jupyter-client 7, please make sure your message is JSON-compliant

/Users/mark/Library/Python/3.10/lib/python/site-packages/jupyter_client/session.py:721: UserWarning:

Message serialization failed with:
Out of range float values are not JSON compliant
Supporting this message is deprecated in jupyter-client 7, please make sure your message is JSON-compliant

/Users/mark/Library/Python/3.10/lib/python/site-packages/jupyter_client/session.py:721: UserWarning:

Message serialization failed with:
Out of range float values are not JSON compliant
Supporting this message is deprecated in jupyter-client 7, please make sure your message is JSON-compliant

/Users/mark/Library/Python/3.10/lib/python/site-packages/jupyter_client/session.py:721: UserWarning:

Message serializ

KeyboardInterrupt: 

Wait expired, Browser is being closed by watchdog.


More documentation can be found from here: 
https://vectorbt.dev/getting-started/features/#plotting

https://github.com/polakowo/vectorbt

# Also interesting blog i found from here https://evanlariviereblog.wordpress.com/2022/03/12/trend-following-asset-class-rotation-in-vectorbt/

# Trend-Following Asset Class Rotation in VectorBT 

You can read the original white paper here:

https://papers.ssrn.com/sol3/papers.cfm?abstract_id=962461

To summarize, the paper looks at a risk-management strategy based on a simple quantitative market-timing model. The paper’s objective is to design a basic trading model that works in the vast majority of markets, where the researchers focused on simplicity over optimization. According to them, market timing can be used as a risk-reduction technique that guides an investor in exiting a potentially risky asset class in favor of risk-free assets. We will explore this in vectorbt with a simple moving-average model that leverages the same parameters across assets and only considers price as an input.

Momentum indicators like moving averages are simple but powerful tools. The moving average itself, is the best available time-domain signal filter. At the same time, it’s also quite possibly the worst filter for frequency domain encoded signals. It utterly lacks the capacity to separate bands of frequencies from one another. This is important because algorithmic trading, specifically returns, are often modelled in a signal processing framework. Other filters, like the Hilbert transform, place emphasis on the time and frequency domain and separate various levels of seasonality and cycles from the original price time series. We will not be using them today, but will consider them in a future article. It’s important to mention this context, because most folks don’t recognize that moving averages are filters that come from our noise reduction signal processing toolkit.

Read more https://evanlariviereblog.wordpress.com/2022/03/12/trend-following-asset-class-rotation-in-vectorbt/ 